# ReportVerifier — Notebook de entrenamiento (Fase 1)

**Objetivo**: entrenar el detector de reportes incoherentes/falsos de AlertaYa y exportar `verifier_v1.joblib` a `../src/models/`.

**Decisiones de arquitectura** (ver memoria del proyecto):
- **ECOD** (PyOD) como primario — paramétrico-cero, robusto a distribuciones no gaussianas por dimensión.
- **Isolation Forest** como segundo voto — captura *interacciones* entre features (el punto ciego de ECOD, que asume independencia).
- **Ensemble AND**: se marca anomalía solo si AMBOS coinciden → menos falsos positivos (clave en una app de seguridad).
- **Encoding cíclico sin/cos** para `hour` y `day_of_week` — crítico para ECOD (23h y 0h son vecinas, no opuestas).

> ⚠️ Datos **sintéticos** para que el pipeline corra end-to-end. Reemplazar por datos abiertos reales de Lima antes de producción.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report, roc_auc_score

from pyod.models.ecod import ECOD

RNG = np.random.default_rng(42)
pd.set_option('display.max_columns', None)
print('Setup OK')

## 1. Datos sintéticos

Generamos reportes "normales" (coherentes con patrones de Lima) y anomalías inyectadas con ground-truth (`is_anomaly`), para poder MEDIR el modelo.


In [ ]:
# Centroides aproximados de distritos de Lima (lat, lng)
DISTRICTS = {
    'Miraflores':              (-12.121, -77.030),
    'Barranco':                (-12.149, -77.022),
    'San Isidro':              (-12.097, -77.036),
    'Surco':                   (-12.135, -76.994),
    'La Victoria':             (-12.066, -77.030),
    'San Juan de Lurigancho':  (-11.985, -77.006),
    'Callao':                  (-12.056, -77.118),
    'Comas':                   (-11.949, -77.061),
}
INCIDENT_TYPES = ['ROBBERY', 'ASSAULT', 'THEFT', 'VANDALISM', 'SUSPICIOUS']
NAMES = list(DISTRICTS.keys())

def _jitter(center, scale=0.01):
    return float(center + RNG.normal(0, scale))

def _night_bias():
    # Crimen sesgado a la noche (20:00-03:00)
    w = np.ones(24)
    for h in range(24):
        if h >= 20 or h <= 3:
            w[h] = 3.0
    return w / w.sum()

def make_normal(n):
    rows = []
    for _ in range(n):
        d = RNG.choice(NAMES)
        lat0, lng0 = DISTRICTS[d]
        rows.append({
            'incident_type': str(RNG.choice(INCIDENT_TYPES, p=[0.35, 0.20, 0.30, 0.10, 0.05])),
            'lat': _jitter(lat0), 'lng': _jitter(lng0),
            'hour': int(RNG.choice(range(24), p=_night_bias())),
            'day_of_week': int(RNG.integers(0, 7)),
            'weapon': str(RNG.choice(['none', 'knife', 'firearm'], p=[0.70, 0.22, 0.08])),
            'injured': str(RNG.choice(['no', 'yes'], p=[0.85, 0.15])),
            'report_count': int(RNG.integers(1, 5)),
            'is_anomaly': 0,
        })
    return rows

def make_anomalies(n):
    rows = []
    for _ in range(n):
        d = RNG.choice(NAMES)
        lat0, lng0 = DISTRICTS[d]
        kind = int(RNG.integers(0, 3))
        if kind == 0:
            # Combinacion incoherente: arma de fuego + 1 solo reporte + mediodia + tipo leve
            row = {'incident_type': 'VANDALISM', 'lat': _jitter(lat0), 'lng': _jitter(lng0),
                   'hour': int(RNG.choice([12, 13, 14])), 'day_of_week': int(RNG.integers(0, 7)),
                   'weapon': 'firearm', 'injured': 'yes', 'report_count': 1}
        elif kind == 1:
            # Coordenadas fuera de Lima (spoof / troll)
            row = {'incident_type': str(RNG.choice(INCIDENT_TYPES)), 'lat': _jitter(-12.0, 0.6), 'lng': _jitter(-77.0, 0.6),
                   'hour': int(RNG.integers(0, 24)), 'day_of_week': int(RNG.integers(0, 7)),
                   'weapon': str(RNG.choice(['none', 'knife', 'firearm'])), 'injured': str(RNG.choice(['no', 'yes'])),
                   'report_count': 1}
        else:
            # report_count absurdamente alto de golpe (ataque coordinado)
            row = {'incident_type': 'SUSPICIOUS', 'lat': _jitter(lat0), 'lng': _jitter(lng0),
                   'hour': int(RNG.integers(0, 24)), 'day_of_week': int(RNG.integers(0, 7)),
                   'weapon': 'none', 'injured': 'no', 'report_count': int(RNG.integers(40, 80))}
        row['is_anomaly'] = 1
        rows.append(row)
    return rows

df = pd.DataFrame(make_normal(2000) + make_anomalies(100))
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print('Shape:', df.shape)
df.head()

## 2. EDA rápida

In [ ]:
print(df['is_anomaly'].value_counts(), '\n')
print(df['incident_type'].value_counts(), '\n')

fig, ax = plt.subplots(1, 2, figsize=(11, 3))
df[df.is_anomaly == 0]['hour'].hist(bins=24, ax=ax[0]); ax[0].set_title('Hora — reportes normales')
df['report_count'].clip(upper=10).hist(bins=10, ax=ax[1]); ax[1].set_title('report_count (clip 10)')
plt.tight_layout(); plt.show()

## 3. Feature engineering

- **Cíclico** sin/cos para `hour` y `day_of_week` (crítico para ECOD; opcional para árboles, pero usamos una sola matriz por simplicidad).
- **Ordinal** para `weapon`/`injured` (tienen orden de gravedad).
- **One-hot** para `incident_type` (categórico sin orden).


In [ ]:
FLAG_MAPS = {
    'weapon':  {'none': 0, 'knife': 1, 'firearm': 2},
    'injured': {'no': 0, 'yes': 1},
}

def build_features(frame, ohe=None, fit=False):
    f = frame.copy()
    # Encoding ciclico
    f['hour_sin'] = np.sin(2 * np.pi * f['hour'] / 24)
    f['hour_cos'] = np.cos(2 * np.pi * f['hour'] / 24)
    f['dow_sin']  = np.sin(2 * np.pi * f['day_of_week'] / 7)
    f['dow_cos']  = np.cos(2 * np.pi * f['day_of_week'] / 7)
    # Flags ordinales
    f['weapon_lvl']  = f['weapon'].map(FLAG_MAPS['weapon'])
    f['injured_lvl'] = f['injured'].map(FLAG_MAPS['injured'])
    # One-hot del tipo
    if fit:
        ohe = OneHotEncoder(categories=[INCIDENT_TYPES], sparse_output=False, handle_unknown='ignore')
        type_arr = ohe.fit_transform(f[['incident_type']])
    else:
        type_arr = ohe.transform(f[['incident_type']])
    type_cols = ['type_' + t for t in INCIDENT_TYPES]
    type_df = pd.DataFrame(type_arr, columns=type_cols, index=f.index)
    num_cols = ['lat', 'lng', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'weapon_lvl', 'injured_lvl', 'report_count']
    X = pd.concat([f[num_cols], type_df], axis=1)
    return X, ohe

X, ohe = build_features(df, fit=True)
FEATURE_COLUMNS = list(X.columns)
print('Features:', FEATURE_COLUMNS)
X.head()

## 4. Entrenar ECOD + Isolation Forest

`contamination` = fracción esperada de anomalías. Lo seteamos en 5% (ajustar con datos reales).


In [ ]:
CONTAMINATION = 0.05

ecod = ECOD(contamination=CONTAMINATION)
ecod.fit(X.values)

iforest = IsolationForest(n_estimators=200, contamination=CONTAMINATION, random_state=42)
iforest.fit(X.values)

print('ECOD + IsolationForest entrenados')

## 5. Ensemble + evaluación

Como los datos son sintéticos tenemos `is_anomaly` real → medimos precision/recall.
El **ensemble AND** prioriza no marcar reportes legítimos como falsos (precision alta en la clase anomalía).


In [ ]:
ecod_pred = ecod.predict(X.values)                          # PyOD: 1 = outlier
if_pred = (iforest.predict(X.values) == -1).astype(int)     # sklearn: -1 = outlier
ensemble = ((ecod_pred == 1) & (if_pred == 1)).astype(int)  # AND

y = df['is_anomaly'].values
print('== ECOD ==');         print(classification_report(y, ecod_pred, digits=3))
print('== IsolationForest =='); print(classification_report(y, if_pred, digits=3))
print('== Ensemble AND ==');  print(classification_report(y, ensemble, digits=3))

print('ROC-AUC ECOD    :', round(roc_auc_score(y, ecod.decision_function(X.values)), 3))
print('ROC-AUC IForest :', round(roc_auc_score(y, -iforest.score_samples(X.values)), 3))

## 6. Función de inferencia (contrato `VerifyReportResponse`)

Espeja el dominio: `is_coherent`, `confidence`, `suggested_severity`. Las **reglas duras** de `CONSTRAINTS.md` (arma de fuego / heridos → CRITICAL) son el piso de seguridad — el modelo NO las reemplaza.


In [ ]:
def suggest_severity(weapon, injured, report_count):
    if weapon == 'firearm' or injured == 'yes':
        return 'CRITICAL'
    if report_count >= 3:
        return 'MODERATE'
    return 'LOW'

def verify_report(report: dict, bundle: dict) -> dict:
    row = pd.DataFrame([report])
    X1, _ = build_features(row, ohe=bundle['ohe'], fit=False)
    X1 = X1[bundle['feature_columns']].values
    ecod_flag = int(bundle['ecod'].predict(X1)[0] == 1)
    if_flag = int(bundle['iforest'].predict(X1)[0] == -1)
    is_anomaly = bool(ecod_flag and if_flag)  # ensemble AND
    # confidence: prob. de outlier de ECOD (min-max sobre scores de train), orientada a la decision
    p_out = float(bundle['ecod'].predict_proba(X1)[0, 1])
    confidence = p_out if is_anomaly else 1.0 - p_out
    return {
        'is_coherent': not is_anomaly,
        'confidence': round(confidence, 3),
        'suggested_severity': suggest_severity(
            report.get('weapon', 'none'), report.get('injured', 'no'), report.get('report_count', 1)
        ),
    }

sample_ok  = {'incident_type': 'ROBBERY',   'lat': -12.121, 'lng': -77.030, 'hour': 22, 'day_of_week': 4, 'weapon': 'knife',   'injured': 'no',  'report_count': 2}
sample_bad = {'incident_type': 'VANDALISM', 'lat': -9.000,  'lng': -70.000, 'hour': 13, 'day_of_week': 2, 'weapon': 'firearm', 'injured': 'yes', 'report_count': 1}
print('definido')

## 7. Exportar artefacto

Bundle único (modelos + encoder + metadata) → `../src/models/verifier_v1.joblib`. `RiskPredictor.load_model()` lo levanta con `joblib.load()`.


In [ ]:
# Resuelve src/models corra desde ml/notebooks/ o desde ml/
HERE = Path.cwd()
ML_ROOT = HERE.parent if HERE.name == 'notebooks' else HERE
MODELS_DIR = ML_ROOT / 'src' / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT = MODELS_DIR / 'verifier_v1.joblib'

bundle = {
    'version': 'v1',
    'ecod': ecod,
    'iforest': iforest,
    'ohe': ohe,
    'feature_columns': FEATURE_COLUMNS,
    'incident_types': INCIDENT_TYPES,
    'contamination': CONTAMINATION,
    'ensemble_rule': 'AND',
    'flag_maps': FLAG_MAPS,
}
joblib.dump(bundle, ARTIFACT)
print('Guardado:', ARTIFACT.resolve())

# Smoke test del artefacto serializado
loaded = joblib.load(ARTIFACT)
print('Normal  ->', verify_report(sample_ok, loaded))
print('Anomalo ->', verify_report(sample_bad, loaded))

## Próximos pasos

1. **Datos reales** de Lima (datos abiertos) en vez de sintéticos → re-entrenar.
2. **Calibrar `confidence`** (hoy es un proxy sigmoide del score ECOD).
3. Mover `build_features` + `verify_report` a la capa `infrastructure`/`application` del servicio (hoy `domain` solo tiene el stub).
4. Wirear el router en `main.py` (`/verify`) y conectar al threshold engine del API.
